**Important Interview Question: How to parse json columns in spark?**
  1. define schema
  2. I will use from_json to parse json data by applying the schema defined (to flatten the semistructured data)
  3. access using .notation (nested columns)
  4. Use explode to transpose the collection data (map or list) eg. from row1- 1,{a:10,b:20} to row1- 1,a,10  then row2- 2,b,20

**Silver Load Starts Here**
1. Reading Bronze ADLS Data

In [0]:
from pyspark.sql.functions import col, regexp_replace, from_json, explode, expr,lit
from pyspark.sql.types import *
from datetime import datetime

from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, explode, col, regexp_replace
from pyspark.sql.types import StructType, StructField, StringType, MapType, DoubleType, LongType

dbutils.widgets.text("filename", "abcdata")
data=dbutils.widgets.get("filename")

# 1. Configuration
adlsacct = 'we47toolstorage'
containername = 'adfcontainer'
accesskey = 'replacewithproperaccesskey'

todaydt=datetime.now().strftime("%Y-%m-%d")
#todaydt='2026-05-09'
print("filename from datafactory? ",data)
print(f"filename with date appended? ",f"{data}_{todaydt}")

spark.conf.set(f"fs.azure.account.key.{adlsacct}.dfs.core.windows.net", accesskey)

path = f"abfss://{containername}@{adlsacct}.dfs.core.windows.net/targetdir/{data}_{todaydt}_*"

# 2. Read CSV (FIXED OPTIONS)
df_raw = spark.read.option("header", "true") \
    .option("multiLine", "true") \
    .option("quote", "\"") \
    .csv(path)
df_raw.show(10,False)
#df_raw_part1=df_raw.repartition(2,col("currencyname")) #coalesce use range partitioning, repartition use round robin partitioning
#

In [0]:
spark.sql("select hash('usd')").show()

2. Cleaning/reformatting/munging json data

In [0]:
# Step 1: Clean JSON
df_clean = df_raw.withColumn(
    "json",
    regexp_replace(col("apiresponsejson"), '^"|"$', ''))

df_clean.cache() #enhancing performance

df_clean = df_clean.withColumn(
    "json", regexp_replace(col("json"), '\\\\\"', '"'))

# Step 2: REMOVE CORRUPTED TAIL
df_clean = df_clean.withColumn(
    "json_valid",
    regexp_replace(col("json"), ',\"ADFHttpStatusCodeInResponse\".*$', '}'))

df_clean.show(3,False)

%md
3. Curation (Flattening, Transposing, Structurizing etc.,)

In [0]:
# Step 3: Schema definition to parse only the required attributes of json as per the business need.
api_schema = StructType([
    StructField("result", StringType(), True),
    StructField("documentation", StringType(), True),
    StructField("terms_of_use", StringType(), True),
    StructField("base_code", StringType(), True),
    StructField("conversion_rates", MapType(StringType(), DoubleType()), True)])

'''{"result":"success",
"documentation":"https://www.exchangerate-api.com/docs",
"terms_of_use":"https://www.exchangerate-api.com/terms",
"base_code":"INR",
"conversion_rates":{"INR":1,"AED":0.03887,"AFN":0.6717,"ALL":0.859,"AMD":3.909,"ANG":0.01895,"AOA":10.0356,"ARS":14.7563,"AUD":0.01461,"AWG":0.01895,"AZN":0.01798,"BAM":0.0176,"BBD":0.02117,"BDT":1.3003}}
'''

display(df_clean.limit(3))

# Step 4: Parse JSON & Data column derivation with minimal json data
df_parsed = df_clean.withColumn(
    "data",
    from_json(col("json_valid"), api_schema)#Interview question to parse json string into structured format, what function you use? from_json(json_string_col,schema)
)
display(df_parsed.limit(3))

# Step 5: Select only required columns and Expand/Structurize/Normalize the json columns alone from the data column
df_final = df_parsed.select(
    "currencyname",
    "data",    
    "data.*")

display(df_final)

df_final1 = df_parsed.select("data.base_code")#How to access the data in the struct column? using '.' notation we can do "data.base_code"
#display(df_final1)

# Step 6: Flatten/transpose (opposite of pivot) the dictionary column into multiple rows
df_flat = df_final.select(
    "base_code",
    explode("conversion_rates").alias("target_currency", "rate"))

df_flat.withColumn("loaddt",lit(todaydt)).write.partitionBy("loaddt","base_code").saveAsTable("default.currency_silver1", mode="append")

display(df_flat)


In [0]:
display(spark.sql("show partitions default.currency_silver1"))